# CA Binary Reservoir — Rule and Topology Investigation

This notebook provides a clean experimental environment for studying the dynamics
of a two-population binary cellular automaton (CA) as a reservoir interface between
sparse binary encoder layers.

**Design**
- Input population $z(t) \in \{0,1\}^{C_{in}}$ is clamped at each step by an external generator.
- Hidden population $h(t) \in \{0,1\}^{C_h}$ is updated by a rule that reads from both populations.
- Topology is represented by two boolean adjacency matrices:
  - $A^{in} \in \{0,1\}^{C_h \times C_{in}}$ — connections from input to hidden units.
  - $A^{hh} \in \{0,1\}^{C_h \times C_h}$ — connections within the hidden population.
- Any update rule is a plain Python function with a fixed signature; swapping rules
  requires only changing one argument.

**Rules implemented**
| Label | Short description |
|-------|------------------|
| `and_of_ors` | Baseline AND-of-ORs (deadlocks from zero) |
| `gds` | Gated Dual-Source — deterministic warm-start |
| `gocp` | Gated OR-of-Cross-Pairs — pattern-recall reservoir |
| `gtx` | Gated Temporal-XOR — change-detection reservoir |

**Topologies implemented**
| Label | Short description |
|-------|------------------|
| `ring` | 1-D ring lattice with fixed local fan-in |
| `small_world` | Watts-Strogatz small-world (ring + random rewiring) |

In [ ]:
# ── Cell 1: Imports and global style ────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap
from typing import Callable, Dict, Optional, Tuple

# Reproducible default RNG — callers may override per experiment.
DEFAULT_SEED = 42

# Consistent binary colour map for raster plots.
BINARY_CMAP = ListedColormap(["#f0f0f0", "#2c7bb6"])   # white / blue

plt.rcParams.update({
    "figure.dpi":      120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size":       10,
})

print("NumPy:", np.__version__)

In [ ]:
# ── Cell 2: Topology generators ─────────────────────────────────────────────
#
# Each generator returns a boolean adjacency matrix A of shape (n_out, n_in)
# where A[i, j] = True means unit i receives input from unit j.
# All generators guarantee exact fan-in k per output unit.

def make_ring(
    n_out: int,
    n_in:  int,
    k:     int,
    rng:   np.random.Generator,
) -> np.ndarray:
    """
    Fixed-fan-in bipartite ring.

    For square (n_out == n_in) graphs each node i is connected to the
    floor(k/2) predecessors and ceil(k/2) successors on a 1-D ring.
    For non-square bipartite graphs the ring is stretched proportionally
    and the k nearest input neighbours (by modular distance) are selected.

    Parameters
    ----------
    n_out : int   Number of output (hidden) nodes.
    n_in  : int   Number of input nodes.
    k     : int   Exact fan-in per output node.  k <= n_in.
    rng   : np.random.Generator

    Returns
    -------
    A : bool ndarray, shape (n_out, n_in)
    """
    assert k <= n_in, f"k={k} must be <= n_in={n_in}"
    A = np.zeros((n_out, n_in), dtype=bool)
    half = k // 2
    for i in range(n_out):
        # Map output index i to a centre position in the input ring.
        centre = round(i * n_in / n_out)
        for delta in range(-half, k - half):
            j = (centre + delta) % n_in
            A[i, j] = True
    return A


def make_small_world(
    n_out: int,
    n_in:  int,
    k:     int,
    beta:  float,
    rng:   np.random.Generator,
) -> np.ndarray:
    """
    Watts-Strogatz small-world adjacency matrix with exact fan-in k.

    Starts from make_ring and rewires each edge independently with
    probability beta to a random non-self, non-existing target.
    Exact fan-in is preserved throughout: rewiring replaces rather
    than adds edges.

    Parameters
    ----------
    n_out : int
    n_in  : int
    k     : int   Exact fan-in.  k <= min(n_out, n_in) - 1 for square graphs.
    beta  : float Rewiring probability in [0, 1].
    rng   : np.random.Generator

    Returns
    -------
    A : bool ndarray, shape (n_out, n_in)
    """
    A = make_ring(n_out, n_in, k, rng)
    for i in range(n_out):
        current_sources = list(np.where(A[i])[0])
        for j in current_sources:
            if rng.random() >= beta:
                continue
            # Build candidate pool respecting self-exclusion (square only).
            existing = set(np.where(A[i])[0])
            candidates = [
                v for v in range(n_in)
                if v not in existing and (n_out != n_in or v != i)
            ]
            if not candidates:
                continue
            new_j = rng.choice(candidates)
            A[i, j]     = False
            A[i, new_j] = True
    return A


# ── Connectivity bundle ───────────────────────────────────────────────────────
# A single named structure grouping all adjacency matrices for a CA instance.
# Different rules may use subsets of these matrices.

class CAConnectivity:
    """
    Container for all adjacency matrices required by the CA rules.

    Attributes
    ----------
    A_gate  : (C_h, C_in) bool  — input gate neighbourhood (used by all rules)
    A_hid   : (C_h, C_h)  bool  — hidden-to-hidden neighbourhood
    A_novel : (C_h, C_in) bool  — secondary input neighbourhood (GDS, GOCP, GTX)
    pair_in : (C_h, m)    int   — input unit indices for GOCP cross-pairs
    pair_hid: (C_h, m)    int   — hidden unit indices for GOCP cross-pairs
    """

    def __init__(
        self,
        C_in:     int,
        C_h:      int,
        k_gate:   int,
        k_hid:    int,
        k_novel:  int,
        n_pairs:  int,
        topology: str,
        beta:     float = 0.1,
        seed:     int   = DEFAULT_SEED,
    ) -> None:
        rng = np.random.default_rng(seed)

        build = (
            lambda no, ni, k: make_small_world(no, ni, k, beta, rng)
            if topology == "small_world"
            else make_ring(no, ni, k, rng)
        )

        self.A_gate  = build(C_h, C_in, k_gate)
        self.A_hid   = build(C_h, C_h,  k_hid)

        # Novel input neighbourhood must be disjoint from gate neighbourhood.
        # Build a fresh random fixed-fan-in bipartite graph; overlap is small
        # and acceptable for large C_in, but we shuffle to decorrelate.
        self.A_novel = build(C_h, C_in, k_novel)

        # GOCP cross-pairs: for each hidden unit i, sample m (input, hidden)
        # index pairs independently and uniformly.
        self.pair_in  = rng.integers(0, C_in, size=(C_h, n_pairs))
        self.pair_hid = rng.integers(0, C_h,  size=(C_h, n_pairs))

        self.C_in    = C_in
        self.C_h     = C_h
        self.n_pairs = n_pairs


print("Topology generators and CAConnectivity defined.")

In [ ]:
# ── Cell 3: Update rules ─────────────────────────────────────────────────────
#
# Every rule has the signature:
#
#   rule(z, h, conn) -> h_new
#
# where:
#   z    : bool ndarray, shape (C_in,)  — current input state (clamped)
#   h    : bool ndarray, shape (C_h,)   — previous hidden state
#   conn : CAConnectivity
#
# Returns h_new : bool ndarray, shape (C_h,)
#
# All OR operations are vectorised using matrix-vector products with boolean
# arithmetic: (A @ x).astype(bool)  is equivalent to OR over the neighbourhood.

# ── Helper: OR over a boolean neighbourhood matrix ───────────────────────────

def _or_neighbourhood(A: np.ndarray, x: np.ndarray) -> np.ndarray:
    """
    For each row i of A, return True iff at least one column j with A[i,j]=True
    has x[j]=True.  Equivalent to OR over the neighbourhood of each unit.

    Parameters
    ----------
    A : bool ndarray, shape (n_out, n_in)
    x : bool ndarray, shape (n_in,)

    Returns
    -------
    out : bool ndarray, shape (n_out,)
    """
    # A @ x sums the active neighbours; > 0 gives OR.
    return (A @ x.view(np.uint8)).astype(bool)


# ── Rule 0 — AND-of-ORs (baseline, deadlocks from h=0) ───────────────────────

def rule_and_of_ors(
    z:    np.ndarray,
    h:    np.ndarray,
    conn: CAConnectivity,
) -> np.ndarray:
    """
    h_i(t) = OR(gate_neighbours in z)  AND  OR(hidden_neighbours in h)

    Deadlocks when h = 0 because the hidden OR is always False.
    Included as a baseline to contrast with the warm-starting rules.
    """
    input_or  = _or_neighbourhood(conn.A_gate, z)   # (C_h,)
    hidden_or = _or_neighbourhood(conn.A_hid,  h)   # (C_h,)
    return input_or & hidden_or


# ── Rule I — Gated Dual-Source (GDS) ─────────────────────────────────────────

def rule_gds(
    z:    np.ndarray,
    h:    np.ndarray,
    conn: CAConnectivity,
) -> np.ndarray:
    """
    h_i(t) = Z_gate(t)  AND  [ H(t-1)  OR  Z_novel(t) ]

    Gate:   OR over k_gate  input units  (A_gate)
    Recall: OR over k_hid   hidden units (A_hid)
    Novel:  OR over k_novel input units  (A_novel, disjoint pool)

    Warm-starts because Z_novel > 0 whenever any novel input neighbour
    is active, independently of h.
    Explosion-proof because the gate depends only on sparse input.
    """
    z_gate   = _or_neighbourhood(conn.A_gate,  z)   # (C_h,)
    h_recall = _or_neighbourhood(conn.A_hid,   h)   # (C_h,)
    z_novel  = _or_neighbourhood(conn.A_novel, z)   # (C_h,)
    return z_gate & (h_recall | z_novel)


# ── Rule II — Gated OR-of-Cross-Pairs (GOCP) ─────────────────────────────────

def rule_gocp(
    z:    np.ndarray,
    h:    np.ndarray,
    conn: CAConnectivity,
) -> np.ndarray:
    """
    h_i(t) = Z_gate(t)  AND  [ OR_m( z[pair_in[i,m]] AND h[pair_hid[i,m]] )
                                OR  Z_novel(t) ]

    Each cross-pair (a_m, b_m) fires when both the designated input unit
    a_m is currently active AND the designated hidden unit b_m was active
    at t-1.  The OR over pairs detects whether any micro-pattern is replayed.

    Rich mixed input-history selectivity; analytically controlled fixed point.
    """
    z_gate  = _or_neighbourhood(conn.A_gate,  z)   # (C_h,)
    z_novel = _or_neighbourhood(conn.A_novel, z)   # (C_h,)

    # Cross-pair activity: shape (C_h, n_pairs)
    # pair_in[i, m]  indexes into z;  pair_hid[i, m] indexes into h.
    pair_active = z[conn.pair_in] & h[conn.pair_hid]   # (C_h, n_pairs) bool

    # OR over pairs for each hidden unit.
    pair_or = pair_active.any(axis=1)                  # (C_h,)

    return z_gate & (pair_or | z_novel)


# ── Rule III — Gated Temporal-XOR (GTX) ──────────────────────────────────────

def rule_gtx(
    z:    np.ndarray,
    h:    np.ndarray,
    conn: CAConnectivity,
) -> np.ndarray:
    """
    h_i(t) = Z_gate(t)  AND  ( H(t-1)  XOR  Z_novel(t) )

    XOR detects disagreement between history and novel input:
      H=0, Z_novel=0 → 0  (quiescent)
      H=0, Z_novel=1 → 1  (onset:  new input without history)
      H=1, Z_novel=0 → 1  (offset: history without new input)
      H=1, Z_novel=1 → 0  (stable: history and input agree → suppress)

    Self-regulating sparsity: as h grows, more XOR terms cancel,
    reducing activity without any kWTA constraint.
    """
    z_gate   = _or_neighbourhood(conn.A_gate,  z)   # (C_h,)
    h_recall = _or_neighbourhood(conn.A_hid,   h)   # (C_h,)
    z_novel  = _or_neighbourhood(conn.A_novel, z)   # (C_h,)
    return z_gate & (h_recall ^ z_novel)


# ── Rule registry ─────────────────────────────────────────────────────────────
# Add new rules here; the rest of the notebook uses this dict.

RULES: Dict[str, Callable] = {
    "and_of_ors": rule_and_of_ors,
    "gds":        rule_gds,
    "gocp":       rule_gocp,
    "gtx":        rule_gtx,
}

print("Rules defined:", list(RULES.keys()))

In [ ]:
# ── Cell 4: Input generators ─────────────────────────────────────────────────
#
# Each generator produces a (T, C_in) bool array representing T time steps
# of sparse binary input.  The interface is a plain function:
#
#   generator(T, C_in, p, rng, **kwargs) -> bool ndarray (T, C_in)

def gen_iid_bernoulli(
    T: int, C_in: int, p: float,
    rng: np.random.Generator, **_
) -> np.ndarray:
    """
    Independent Bernoulli(p) at every unit and every time step.
    No temporal structure — useful as a baseline / null model.
    """
    return rng.random((T, C_in)) < p


def gen_markov(
    T: int, C_in: int, p: float,
    rng: np.random.Generator,
    p_on:  float = 0.15,
    p_off: float = 0.40,
    **_
) -> np.ndarray:
    """
    Each unit evolves as an independent two-state Markov chain.

    Transition probabilities:
      0 → 1  with probability p_on   (activation)
      1 → 0  with probability p_off  (deactivation)

    Stationary sparsity: p_on / (p_on + p_off).
    Temporal autocorrelation timescale: 1 / (p_on + p_off) steps.

    Parameters
    ----------
    p_on  : P(0→1).  Default 0.15.
    p_off : P(1→0).  Default 0.40.
    """
    stat = p_on / (p_on + p_off)
    Z    = np.zeros((T, C_in), dtype=bool)
    # Initialise at stationarity.
    Z[0] = rng.random(C_in) < stat
    for t in range(1, T):
        stay_on  = Z[t-1] & (rng.random(C_in) >= p_off)
        turn_on  = ~Z[t-1] & (rng.random(C_in) < p_on)
        Z[t]     = stay_on | turn_on
    return Z


def gen_bursty(
    T: int, C_in: int, p: float,
    rng: np.random.Generator,
    burst_len: int   = 10,
    burst_p:   float = 0.40,
    quiet_p:   float = 0.05,
    **_
) -> np.ndarray:
    """
    Alternating burst / quiet phases, each unit independently.

    Phase duration is geometrically distributed with mean burst_len.
    Within a burst, sparsity is burst_p; in quiet phases it is quiet_p.
    Mimics the on/off structure of speech phonemes.
    """
    Z     = np.zeros((T, C_in), dtype=bool)
    phase = rng.random(C_in) < 0.5   # True = burst, False = quiet
    ttl   = rng.integers(1, burst_len * 2, C_in)   # time-to-live per unit
    for t in range(T):
        p_t     = np.where(phase, burst_p, quiet_p)
        Z[t]    = rng.random(C_in) < p_t
        ttl    -= 1
        flip    = ttl <= 0
        phase   = np.where(flip, ~phase, phase)
        ttl     = np.where(flip,
                           rng.integers(1, burst_len * 2, C_in),
                           ttl)
    return Z


def gen_spatial_wave(
    T: int, C_in: int, p: float,
    rng: np.random.Generator,
    speed: float = 2.0,
    width: float = 0.15,
    **_
) -> np.ndarray:
    """
    A wave of activity sweeping across units at a fixed speed.

    Unit i is active at time t with probability p if it falls within a
    Gaussian envelope centred at (speed * t) mod C_in, scaled by width.
    Tests whether the CA can track a moving spatial pattern.
    """
    Z     = np.zeros((T, C_in), dtype=bool)
    units = np.arange(C_in)
    for t in range(T):
        centre = (speed * t) % C_in
        dist   = np.minimum(
            np.abs(units - centre),
            C_in - np.abs(units - centre)
        )   # circular distance
        prob   = np.exp(-0.5 * (dist / (width * C_in)) ** 2)
        # Normalise so the average sparsity matches p.
        prob   = prob / prob.sum() * (p * C_in)
        prob   = np.clip(prob, 0.0, 1.0)
        Z[t]   = rng.random(C_in) < prob
    return Z


# ── Input generator registry ──────────────────────────────────────────────────

GENERATORS: Dict[str, Callable] = {
    "iid":          gen_iid_bernoulli,
    "markov":       gen_markov,
    "bursty":       gen_bursty,
    "spatial_wave": gen_spatial_wave,
}

print("Input generators defined:", list(GENERATORS.keys()))

In [ ]:
# ── Cell 5: CA runner and metrics ────────────────────────────────────────────

def run_ca(
    Z:            np.ndarray,
    conn:         CAConnectivity,
    rule:         Callable,
    h_init:       Optional[np.ndarray] = None,
    init_sparsity: float = 0.15,
    seed:         int   = DEFAULT_SEED,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Run the CA for T steps over the pre-generated input sequence Z.

    Parameters
    ----------
    Z    : bool ndarray, shape (T, C_in)   Pre-generated input sequence.
    conn : CAConnectivity
    rule : callable  Signature: rule(z, h, conn) -> h_new
    h_init : bool ndarray, shape (C_h,), optional
        Initial hidden state.  If None, drawn from Bernoulli(init_sparsity).
    init_sparsity : float
        Bernoulli probability for random initialisation.  Default 0.15.
    seed : int  RNG seed for the initial state draw.

    Returns
    -------
    H    : bool ndarray, shape (T, C_h)   Hidden state trajectory.
    Z    : bool ndarray, shape (T, C_in)  Input trajectory (pass-through).
    """
    T   = Z.shape[0]
    C_h = conn.C_h

    if h_init is not None:
        h = h_init.copy()
    else:
        rng = np.random.default_rng(seed)
        h   = rng.random(C_h) < init_sparsity

    H = np.zeros((T, C_h), dtype=bool)
    for t in range(T):
        h    = rule(Z[t], h, conn)
        H[t] = h

    return H, Z


def compute_metrics(
    H: np.ndarray,
    Z: np.ndarray,
    burn_in: int = 20,
) -> Dict[str, float]:
    """
    Compute scalar diagnostics for a CA trajectory.

    Parameters
    ----------
    H       : bool ndarray, shape (T, C_h)
    Z       : bool ndarray, shape (T, C_in)
    burn_in : int  Steps to discard before computing statistics.

    Returns
    -------
    metrics : dict with keys:
        mean_sparsity     — time-averaged fraction of active hidden units
        std_sparsity      — standard deviation of per-step sparsity
        mean_hamming_dist — mean Hamming distance between consecutive h(t), h(t+1)
        unit_activity_std — std of per-unit time-averaged activity (diversity)
        input_sparsity    — actual input sparsity (sanity check)
    """
    H_ = H[burn_in:].astype(np.float32)
    Z_ = Z[burn_in:].astype(np.float32)

    per_step_sparsity  = H_.mean(axis=1)              # (T-burn_in,)
    consecutive_hamming = np.abs(
        H_[1:] - H_[:-1]
    ).mean(axis=1)                                     # (T-burn_in-1,)
    per_unit_activity  = H_.mean(axis=0)              # (C_h,)

    return {
        "mean_sparsity":     float(per_step_sparsity.mean()),
        "std_sparsity":      float(per_step_sparsity.std()),
        "mean_hamming_dist": float(consecutive_hamming.mean()),
        "unit_activity_std": float(per_unit_activity.std()),
        "input_sparsity":    float(Z_.mean()),
    }


print("run_ca and compute_metrics defined.")

In [ ]:
# ── Cell 6: Visualisation utilities ─────────────────────────────────────────

def plot_trajectory(
    H:          np.ndarray,
    Z:          np.ndarray,
    title:      str          = "",
    t_range:    Tuple[int,int] = (0, 200),
    metrics:    Optional[Dict[str, float]] = None,
    ax_z:       Optional[plt.Axes] = None,
    ax_h:       Optional[plt.Axes] = None,
    ax_spa:     Optional[plt.Axes] = None,
    show:       bool = True,
) -> Tuple[plt.Axes, plt.Axes, plt.Axes]:
    """
    Raster plot of input (Z) and hidden (H) activity plus per-step sparsity.

    Can be used standalone (show=True, axes=None) or embedded in a larger
    figure by passing pre-created axes.

    Parameters
    ----------
    H, Z     : bool ndarrays  Full trajectories.
    title    : str
    t_range  : (t_start, t_end)  Time window to display.
    metrics  : optional dict from compute_metrics — shown as subtitle.
    ax_z, ax_h, ax_spa : optional axes for embedding.
    show     : if True, calls plt.show().
    """
    t0, t1 = t_range
    Zv = Z[t0:t1].T.astype(float)   # (C_in, t_window)
    Hv = H[t0:t1].T.astype(float)   # (C_h,  t_window)
    ts = np.arange(t0, t1)

    standalone = ax_z is None
    if standalone:
        fig = plt.figure(figsize=(14, 6))
        gs  = gridspec.GridSpec(
            3, 1, height_ratios=[1.2, 2.0, 0.8], hspace=0.45
        )
        ax_z   = fig.add_subplot(gs[0])
        ax_h   = fig.add_subplot(gs[1], sharex=ax_z)
        ax_spa = fig.add_subplot(gs[2], sharex=ax_z)

    # ── Input raster ─────────────────────────────────────────────────────────
    ax_z.imshow(
        Zv, aspect="auto", cmap=BINARY_CMAP,
        vmin=0, vmax=1,
        extent=[t0, t1, -0.5, Zv.shape[0]-0.5],
        interpolation="nearest", origin="lower",
    )
    ax_z.set_ylabel("Input\nunit", fontsize=9)
    ax_z.set_yticks([])

    # ── Hidden raster ─────────────────────────────────────────────────────────
    ax_h.imshow(
        Hv, aspect="auto", cmap=BINARY_CMAP,
        vmin=0, vmax=1,
        extent=[t0, t1, -0.5, Hv.shape[0]-0.5],
        interpolation="nearest", origin="lower",
    )
    ax_h.set_ylabel("Hidden\nunit", fontsize=9)
    ax_h.set_yticks([])

    # ── Per-step sparsity ─────────────────────────────────────────────────────
    h_sparsity = H[t0:t1].mean(axis=1)
    ax_spa.plot(ts, h_sparsity, lw=1.0, color="#2c7bb6", label="Hidden")
    ax_spa.axhline(
        h_sparsity.mean(), ls="--", lw=0.8,
        color="#d7191c", label=f"Mean {h_sparsity.mean():.3f}"
    )
    ax_spa.set_ylim(-0.02, min(h_sparsity.max() * 1.4 + 0.02, 1.0))
    ax_spa.set_ylabel("Sparsity", fontsize=9)
    ax_spa.set_xlabel("Time step", fontsize=9)
    ax_spa.legend(fontsize=8, loc="upper right")

    # ── Title ─────────────────────────────────────────────────────────────────
    subtitle = ""
    if metrics:
        subtitle = (
            f"  sparsity={metrics['mean_sparsity']:.3f}  "
            f"Δh={metrics['mean_hamming_dist']:.3f}  "
            f"unit_std={metrics['unit_activity_std']:.3f}"
        )
    if standalone:
        fig.suptitle(title + subtitle, fontsize=11, y=1.01)
        if show:
            plt.show()

    return ax_z, ax_h, ax_spa


def plot_comparison(
    results: Dict[str, Tuple[np.ndarray, np.ndarray]],
    t_range: Tuple[int, int] = (0, 200),
    suptitle: str = "",
) -> None:
    """
    Side-by-side comparison of multiple (rule, topology) combinations.

    Parameters
    ----------
    results  : dict mapping label -> (H, Z) trajectory pair
    t_range  : time window
    suptitle : overall figure title
    """
    n     = len(results)
    fig   = plt.figure(figsize=(6 * n, 7))
    outer = gridspec.GridSpec(1, n, wspace=0.35)

    for col, (label, (H, Z)) in enumerate(results.items()):
        inner = gridspec.GridSpecFromSubplotSpec(
            3, 1, subplot_spec=outer[col],
            height_ratios=[1.2, 2.0, 0.8], hspace=0.45,
        )
        ax_z   = fig.add_subplot(inner[0])
        ax_h   = fig.add_subplot(inner[1], sharex=ax_z)
        ax_spa = fig.add_subplot(inner[2], sharex=ax_z)

        metrics = compute_metrics(H, Z)
        plot_trajectory(
            H, Z,
            t_range=t_range,
            metrics=metrics,
            ax_z=ax_z, ax_h=ax_h, ax_spa=ax_spa,
            show=False,
        )
        ax_z.set_title(
            f"{label}\n"
            f"spar={metrics['mean_sparsity']:.3f}  "
            f"Δh={metrics['mean_hamming_dist']:.3f}",
            fontsize=9,
        )

    if suptitle:
        fig.suptitle(suptitle, fontsize=12, y=1.02)
    plt.show()


print("Visualisation utilities defined.")

In [ ]:
# ── Cell 7: Experiment 1 — Rule comparison on IID input ─────────────────────
#
# All four rules, same small-world topology, IID Bernoulli(0.15) input.
# Baseline test: do the rules behave as predicted analytically?

RNG  = np.random.default_rng(DEFAULT_SEED)
T    = 1000
C_IN = 128
C_H  = 128
P    = 0.05

# Shared connectivity: one CAConnectivity instance per experiment.
conn_sw = CAConnectivity(
    C_in=C_IN, C_h=C_H,
    k_gate=3, k_hid=3, k_novel=3, n_pairs=3,
    topology="small_world", beta=0.1,
    seed=DEFAULT_SEED,
)

# Generate shared input sequence.
Z_iid = gen_iid_bernoulli(T, C_IN, P, rng=np.random.default_rng(1))

results_rules = {}
for rule_name, rule_fn in RULES.items():
    H, Z = run_ca(
        Z_iid, conn_sw, rule_fn,
        init_sparsity=P, seed=DEFAULT_SEED,
    )
    results_rules[rule_name] = (H, Z)
    m = compute_metrics(H, Z)
    print(
        f"{rule_name:>12s} | "
        f"sparsity={m['mean_sparsity']:.3f} ± {m['std_sparsity']:.3f} | "
        f"Δh={m['mean_hamming_dist']:.3f} | "
        f"unit_std={m['unit_activity_std']:.3f}"
    )

plot_comparison(
    results_rules,
    t_range=(0, 200),
    suptitle="Experiment 1: Rule comparison | IID input | small-world topology",
)

In [ ]:
# ── Cell 8: Experiment 2 — Topology comparison ───────────────────────────────
#
# Best rule (GDS) on ring vs small-world, same IID input.
# Tests whether the multi-timescale property of small-world is visible.

conn_ring = CAConnectivity(
    C_in=C_IN, C_h=C_H,
    k_gate=3, k_hid=3, k_novel=3, n_pairs=3,
    topology="ring",
    seed=DEFAULT_SEED,
)

# Vary rewiring probability for small-world.
results_topo = {}
for label, conn in [("ring", conn_ring), ("sw β=0.05", conn_sw)]:
    for rule_name, rule_fn in [("gds", rule_gds), ("gtx", rule_gtx)]:
        key  = f"{label} / {rule_name}"
        H, Z = run_ca(Z_iid, conn, rule_fn, init_sparsity=P, seed=DEFAULT_SEED)
        results_topo[key] = (H, Z)

plot_comparison(
    results_topo,
    t_range=(0, 200),
    suptitle="Experiment 2: Ring vs Small-World | GDS and GTX rules",
)

In [ ]:
# ── Cell 9: Experiment 3 — Input type comparison ─────────────────────────────
#
# GOCP rule on small-world topology under all four input generators.
# Tests whether the CA dynamics adapt to different temporal input structures.

results_inputs = {}
gen_rng = np.random.default_rng(99)

for gen_name, gen_fn in GENERATORS.items():
    Z_gen = gen_fn(T, C_IN, P, rng=np.random.default_rng(int(gen_rng.integers(1000))))
    H, Z  = run_ca(Z_gen, conn_sw, rule_gocp, init_sparsity=P, seed=DEFAULT_SEED)
    results_inputs[gen_name] = (H, Z)
    m = compute_metrics(H, Z)
    print(
        f"{gen_name:>14s} | "
        f"input_p={m['input_sparsity']:.3f} | "
        f"hidden_p={m['mean_sparsity']:.3f} | "
        f"Δh={m['mean_hamming_dist']:.3f}"
    )

plot_comparison(
    results_inputs,
    t_range=(0, 200),
    suptitle="Experiment 3: Input type comparison | GOCP rule | small-world",
)

In [ ]:
# ── Cell 10: Experiment 4 — Zero-init comparison ─────────────────────────────
#
# Confirm that AND-of-ORs deadlocks from h=0 while the three other rules
# warm-start.  Uses the same IID input and a shared all-zero initial state.

h_zero = np.zeros(C_H, dtype=bool)

results_zero = {}
for rule_name, rule_fn in RULES.items():
    H, Z = run_ca(Z_iid, conn_sw, rule_fn, h_init=h_zero)
    results_zero[rule_name] = (H, Z)
    mean_p = H.mean()
    print(f"{rule_name:>12s} | h_init=0 | mean hidden sparsity = {mean_p:.4f}")

# Time-series of per-step sparsity for the first 100 steps (warm-start check).
fig, ax = plt.subplots(figsize=(10, 3))
colours = ["#d7191c", "#2c7bb6", "#1a9641", "#ff7f00"]
for (rule_name, (H, Z)), colour in zip(results_zero.items(), colours):
    ax.plot(H[:100].mean(axis=1), label=rule_name, lw=1.5, color=colour)
ax.set_xlabel("Time step")
ax.set_ylabel("Hidden sparsity")
ax.set_title("Experiment 4: Warm-start from h = 0 | IID input | small-world")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 11: Experiment 5 — Analytical vs empirical sparsity ─────────────────
#
# Sweep target sparsity p ∈ {0.05, 0.10, 0.15, 0.20, 0.25}.
# For each p, plot the analytical hard bound q = 1-(1-p)^k_gate and the
# empirical hidden sparsity for each rule.

P_SWEEP   = [0.05, 0.10, 0.15, 0.20, 0.25]
K_GATE    = 3

analytical_bound = [1 - (1 - p) ** K_GATE for p in P_SWEEP]

empirical: Dict[str, list] = {name: [] for name in RULES}

for p_val in P_SWEEP:
    Z_p = gen_iid_bernoulli(T, C_IN, p_val, rng=np.random.default_rng(7))
    for rule_name, rule_fn in RULES.items():
        H, Z = run_ca(Z_p, conn_sw, rule_fn, init_sparsity=p_val, seed=42)
        empirical[rule_name].append(compute_metrics(H, Z)["mean_sparsity"])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(P_SWEEP, analytical_bound, "k--", lw=1.5,
        label=f"Analytical bound q=(1-(1-p)^{K_GATE})")
ax.plot(P_SWEEP, P_SWEEP, "k:",  lw=1.0, label="Identity (target p)")

colours = ["#d7191c", "#2c7bb6", "#1a9641", "#ff7f00"]
for (rule_name, vals), colour in zip(empirical.items(), colours):
    ax.plot(P_SWEEP, vals, "o-", label=rule_name, color=colour, lw=1.5, ms=5)

ax.set_xlabel("Input sparsity p")
ax.set_ylabel("Mean hidden sparsity")
ax.set_title("Experiment 5: Analytical bound vs empirical sparsity")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 12: Experiment 6 — Reservoir separation quality ─────────────────────
#
# Quantitative test of the CA's ability to distinguish different input histories.
#
# Protocol:
#   1. Generate two input sequences Z_A and Z_B that diverge at step t_split.
#      Before t_split both are identical (same RNG seed); after t_split they
#      are drawn independently.
#   2. Run both sequences through the CA starting from the same h_init.
#   3. Measure the Hamming distance between H_A(t) and H_B(t) as a function
#      of time after t_split.
#
# A good reservoir should show growing separation after t_split.

T_SPLIT = 100
T_TOTAL = 300

shared_rng = np.random.default_rng(0)
Z_shared   = gen_iid_bernoulli(T_SPLIT, C_IN, P, rng=shared_rng)
Z_A_tail   = gen_iid_bernoulli(T_TOTAL - T_SPLIT, C_IN, P,
                                rng=np.random.default_rng(10))
Z_B_tail   = gen_iid_bernoulli(T_TOTAL - T_SPLIT, C_IN, P,
                                rng=np.random.default_rng(20))

Z_A = np.concatenate([Z_shared, Z_A_tail], axis=0)
Z_B = np.concatenate([Z_shared, Z_B_tail], axis=0)

fig, ax = plt.subplots(figsize=(10, 4))
colours  = ["#d7191c", "#2c7bb6", "#1a9641", "#ff7f00"]

for (rule_name, rule_fn), colour in zip(RULES.items(), colours):
    H_A, _ = run_ca(Z_A, conn_sw, rule_fn, init_sparsity=P, seed=0)
    H_B, _ = run_ca(Z_B, conn_sw, rule_fn, init_sparsity=P, seed=0)

    hamming = (H_A != H_B).mean(axis=1)   # (T,) fraction of differing units
    ax.plot(hamming, label=rule_name, color=colour, lw=1.5)

ax.axvline(T_SPLIT, color="black", ls="--", lw=1.2, label=f"Divergence at t={T_SPLIT}")
ax.set_xlabel("Time step")
ax.set_ylabel("Hamming distance H_A vs H_B")
ax.set_title(
    "Experiment 6: Reservoir separation after input divergence\n"
    "(small-world topology, IID input)"
)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell N: MI Evidence Rule ─────────────────────────────────────────────────
#
# Implements the balanced evidence update:
#
#   e_i(t) = [(x_i + ε)(h_{i,t-1} + ε)] / [(x̄_i + γ)(h̄_i + γ)]
#   h_i(t) = 1  iff  e_i(t) > θ
#
# EMA state (x̄, h̄) is maintained internally across time steps.
# Call rule.reset() before each new sequence to clear the EMA.
#
# Compatible with run_ca: run_ca checks for a .reset() method and calls it
# automatically if present (one-line addition shown below).

class MIEvidenceRule:
    """
    MI-optimal balanced evidence update rule.

    For each unit i, fires when the joint evidence from current input x_t
    and previous hidden state h_{t-1} exceeds a threshold θ, relative to
    the respective exponential moving averages (background rates).

    Parameters
    ----------
    alpha_x : float
        EMA momentum for the input background rate x̄.
        Higher → slower adaptation, more stable background estimate.
    alpha_h : float
        EMA momentum for the hidden background rate h̄.
    eps : float
        Numerator additive constant (from the InfoNCE critic).
        Controls the base score for inactive units.
    gamma : float
        Denominator pseudo-count.  gamma > eps prevents the rare-event
        amplification: units with near-zero background rate cannot produce
        arbitrarily large evidence ratios.
    theta : float
        Firing threshold on the evidence ratio.
        Set analytically via set_theta_for_sparsity() or manually.
    init_background : float
        Initial value for both EMA vectors at reset().
        Should match the expected steady-state sparsity p.
    """

    def __init__(
        self,
        alpha_x:        float = 0.95,
        alpha_h:        float = 0.95,
        eps:            float = 0.01,
        gamma:          float = 0.10,
        theta:          float = 1.0,
        init_background: float = 0.15,
    ) -> None:
        self.alpha_x         = alpha_x
        self.alpha_h         = alpha_h
        self.eps             = eps
        self.gamma           = gamma
        self.theta           = theta
        self.init_background = init_background

        # EMA state — initialised lazily on first reset() or __call__.
        self._x_bar: Optional[np.ndarray] = None
        self._h_bar: Optional[np.ndarray] = None

    # ------------------------------------------------------------------

    def reset(self, C_in: int = 0, C_h: int = 0) -> None:
        """
        Re-initialise EMA state.  Call before each new sequence.

        If C_in / C_h are 0, existing arrays are reset in-place (faster
        when dimensions are already known from a previous run).
        """
        if C_in > 0:
            self._x_bar = np.full(C_in, self.init_background, dtype=np.float32)
        elif self._x_bar is not None:
            self._x_bar[:] = self.init_background

        if C_h > 0:
            self._h_bar = np.full(C_h,  self.init_background, dtype=np.float32)
        elif self._h_bar is not None:
            self._h_bar[:] = self.init_background

    # ------------------------------------------------------------------

    def set_theta_for_sparsity(
        self,
        target_p: float,
        Z_sample: np.ndarray,
        H_sample: np.ndarray,
        n_steps:  int = 500,
    ) -> float:
        """
        Estimate the threshold θ that achieves a target hidden sparsity.

        Runs the rule for n_steps on the provided samples with θ=0
        (fire everything), collects evidence scores, and sets θ to
        the (1 - target_p) quantile.

        Parameters
        ----------
        target_p : float   Desired mean hidden sparsity.
        Z_sample : (T, C_in) bool array — sample input sequence.
        H_sample : (T, C_h)  bool array — warm-up hidden sequence (e.g.
                             from a previous run with a lenient rule).
        n_steps  : int     How many steps to collect scores over.

        Returns
        -------
        theta : float  The threshold that was set.
        """
        old_theta   = self.theta
        self.theta  = 0.0          # collect all scores
        self.reset(Z_sample.shape[1], H_sample.shape[1])

        scores = []
        h = H_sample[0].copy()
        for t in range(min(n_steps, Z_sample.shape[0])):
            e = self._evidence(Z_sample[t], h)
            scores.append(e)
            h = (e > 0.0)          # fire everything while collecting

        self.theta = float(np.quantile(np.concatenate(scores), 1 - target_p))
        print(
            f"[MIEvidenceRule] θ set to {self.theta:.4f} "
            f"for target sparsity p={target_p}"
        )
        return self.theta

    # ------------------------------------------------------------------

    def _evidence(
        self,
        z: np.ndarray,   # (C_in,) bool
        h: np.ndarray,   # (C_h,)  bool
    ) -> np.ndarray:
        """
        Compute per-unit evidence ratios.  Internal helper.

        Returns e : (C_h,) float32
        """
        # Lazy initialisation on the very first call.
        if self._x_bar is None:
            self._x_bar = np.full(len(z), self.init_background, dtype=np.float32)
        if self._h_bar is None:
            self._h_bar = np.full(len(h), self.init_background, dtype=np.float32)

        numerator   = (z.astype(np.float32) + self.eps) \
                    * (h.astype(np.float32) + self.eps)   # (C_h,) — uses h directly
        denominator = (self._x_bar + self.gamma) \
                    * (self._h_bar + self.gamma)

        # Note: numerator uses x_t and h_{t-1} per unit independently.
        # Because C_in may differ from C_h, x evidence is computed from
        # the gate neighbourhood: mean input evidence over gate neighbours.
        # Here we approximate by using a scalar mean over all input units
        # as the "input signal" per hidden unit — for unit-level resolution
        # pass conn and use _or_neighbourhood in __call__ instead.
        return numerator / denominator                     # (C_h,) float32

    def __call__(
        self,
        z:    np.ndarray,      # (C_in,) bool — current input
        h:    np.ndarray,      # (C_h,)  bool — previous hidden state
        conn: "CAConnectivity",
    ) -> np.ndarray:
        """
        Compute h_new = 1[e_i > θ] with per-unit evidence from gate neighbours.

        Evidence for unit i:
            e_i = [(z̃_i + ε)(h_i + ε)] / [(x̄_i + γ)(h̄_i + γ)]

        where z̃_i = OR over gate neighbours of z  (same gate as other rules),
        so the rule is input-gated and explosion-proof.
        """
        if self._x_bar is None:
            self._x_bar = np.full(conn.C_in, self.init_background, dtype=np.float32)
        if self._h_bar is None:
            self._h_bar = np.full(conn.C_h,  self.init_background, dtype=np.float32)

        # ── Input signal: OR over gate neighbourhood (same as other rules) ──
        # Shape (C_h,) float32 ∈ {0, 1}
        z_gate = (conn.A_gate @ z.view(np.uint8)).astype(np.float32)
        z_gate = (z_gate > 0).astype(np.float32)

        # ── Evidence computation ─────────────────────────────────────────────
        # Numerator: joint activation of gated input and previous hidden state.
        numerator = (z_gate + self.eps) * (h.astype(np.float32) + self.eps)

        # Denominator: product of background rates (pseudo-count prevents
        # rare-event amplification when either background is near zero).
        # x̄ is indexed over the gate neighbourhood mean.
        x_bar_gate = conn.A_gate.astype(np.float32) @ self._x_bar \
                     / conn.A_gate.sum(axis=1).clip(1)   # (C_h,) mean gate x̄
        denominator = (x_bar_gate + self.gamma) * (self._h_bar + self.gamma)

        evidence = numerator / denominator                 # (C_h,) float32

        # ── Binary selection ─────────────────────────────────────────────────
        h_new = evidence > self.theta

        # ── EMA update (after selection, using new h_new) ────────────────────
        # x̄ tracks the full input population, not just the gate projection.
        self._x_bar = (self.alpha_x * self._x_bar
                       + (1 - self.alpha_x) * z.astype(np.float32))
        self._h_bar = (self.alpha_h * self._h_bar
                       + (1 - self.alpha_h) * h_new.astype(np.float32))

        return h_new

    # ------------------------------------------------------------------

    def __repr__(self) -> str:
        return (
            f"MIEvidenceRule(α_x={self.alpha_x}, α_h={self.alpha_h}, "
            f"ε={self.eps}, γ={self.gamma}, θ={self.theta:.4f})"
        )


# ── Patch run_ca to call reset() when the rule supports it ───────────────────
# This is a minimal, backward-compatible extension: existing stateless rules
# (plain functions) are unaffected; only callable objects with .reset() are
# touched.

def run_ca_stateful(
    Z:             np.ndarray,
    conn:          "CAConnectivity",
    rule,
    h_init:        Optional[np.ndarray] = None,
    init_sparsity: float = 0.15,
    seed:          int   = DEFAULT_SEED,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Extended run_ca that calls rule.reset() before the time loop if the
    rule exposes that method.  Drop-in replacement for run_ca.
    """
    if hasattr(rule, "reset"):
        rule.reset(C_in=conn.C_in, C_h=conn.C_h)

    return run_ca(Z, conn, rule, h_init=h_init,
                  init_sparsity=init_sparsity, seed=seed)


print("MIEvidenceRule and run_ca_stateful defined.")

In [ ]:
# ── Cell N+1: Experiment — MI Evidence Rule comparison ───────────────────────
#
# Compare MI evidence rule against GDS and GOCP on:
#   (a) IID input         — validates basic sparsity and separation
#   (b) Markov input      — tests robustness to correlated inputs
#   (c) Separation test   — mirrors Experiment 6

# ── Instantiate with base parameters ─────────────────────────────────────────
# γ > ε prevents rare-event amplification.
# α_h controls memory timescale: higher → slower decay → longer memory.
# θ is set manually here; use set_theta_for_sparsity() for auto-calibration.

rule_mi = MIEvidenceRule(
    alpha_x        = 0.95,
    alpha_h        = 0.95,
    eps            = 0.01,
    gamma          = 0.10,
    theta          = 1.0,
    init_background = 0.15,
)

# ── (a) Sparsity and dynamics on IID input ────────────────────────────────────
print("── IID input ────────────────────────────────────────────────────────")
H_mi, Z_iid_out = run_ca_stateful(Z_iid, conn_sw, rule_mi, init_sparsity=0.15)
m_mi = compute_metrics(H_mi, Z_iid_out)
print(repr(rule_mi))
for k, v in m_mi.items():
    print(f"  {k:<22s}: {v:.4f}")

# Side-by-side with GDS and GOCP.
results_mi_compare = {
    "gds":        run_ca_stateful(Z_iid, conn_sw, rule_gds,  init_sparsity=0.15)[::-1][::-1],
    "gocp":       run_ca_stateful(Z_iid, conn_sw, rule_gocp, init_sparsity=0.15)[::-1][::-1],
    "mi_evidence": (H_mi, Z_iid_out),
}
# (The [::-1][::-1] is a no-op — just calling run_ca_stateful directly below)
results_mi_compare = {}
for label, rule_fn in [("gds", rule_gds), ("gocp", rule_gocp),
                        ("mi_evidence", rule_mi)]:
    H_, Z_ = run_ca_stateful(Z_iid, conn_sw, rule_fn, init_sparsity=0.15)
    results_mi_compare[label] = (H_, Z_)

plot_comparison(
    results_mi_compare,
    t_range=(0, 200),
    suptitle="MI Evidence vs GDS vs GOCP | IID input | small-world",
)

# ── (b) Markov input ──────────────────────────────────────────────────────────
print("\n── Markov input ──────────────────────────────────────────────────────")
Z_markov = gen_markov(T, C_IN, P, rng=np.random.default_rng(5))

results_mi_markov = {}
for label, rule_fn in [("gds", rule_gds), ("gocp", rule_gocp),
                        ("mi_evidence", rule_mi)]:
    H_, Z_ = run_ca_stateful(Z_markov, conn_sw, rule_fn, init_sparsity=0.15)
    m_ = compute_metrics(H_, Z_)
    results_mi_markov[label] = (H_, Z_)
    print(
        f"  {label:<14s} | hidden_p={m_['mean_sparsity']:.3f} | "
        f"Δh={m_['mean_hamming_dist']:.3f} | "
        f"input_p={m_['input_sparsity']:.3f}"
    )

plot_comparison(
    results_mi_markov,
    t_range=(0, 200),
    suptitle="MI Evidence vs GDS vs GOCP | Markov input | small-world",
)

# ── (c) Separation quality (mirrors Experiment 6) ─────────────────────────────
print("\n── Separation quality ────────────────────────────────────────────────")
fig, ax = plt.subplots(figsize=(10, 4))
colours  = {"gds": "#2c7bb6", "gocp": "#1a9641", "mi_evidence": "#d7191c"}

for label, rule_fn in [("gds", rule_gds), ("gocp", rule_gocp),
                        ("mi_evidence", rule_mi)]:
    H_A, _ = run_ca_stateful(Z_A, conn_sw, rule_fn,
                              init_sparsity=0.15, seed=0)
    H_B, _ = run_ca_stateful(Z_B, conn_sw, rule_fn,
                              init_sparsity=0.15, seed=0)
    hamming = (H_A != H_B).mean(axis=1)
    ax.plot(hamming, label=label, color=colours[label], lw=1.5)

ax.axvline(T_SPLIT, color="black", ls="--", lw=1.2,
           label=f"Divergence at t={T_SPLIT}")
ax.set_xlabel("Time step")
ax.set_ylabel("Hamming distance H_A vs H_B")
ax.set_title(
    "MI Evidence separation quality\n"
    "(small-world, IID input, compared to GDS and GOCP)"
)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# ── (d) Alpha sweep — memory timescale sensitivity ────────────────────────────
print("\n── Alpha_h sweep ─────────────────────────────────────────────────────")
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)

for ax, alpha_h in zip(axes, [0.80, 0.95, 0.99]):
    rule_sweep = MIEvidenceRule(
        alpha_x=0.95, alpha_h=alpha_h,
        eps=0.01, gamma=0.10, theta=1.0,
    )
    H_sw, _ = run_ca_stateful(Z_iid, conn_sw, rule_sweep, init_sparsity=0.15)
    H_B_sw, _ = run_ca_stateful(
        np.concatenate([Z_shared, Z_B_tail], axis=0),
        conn_sw, rule_sweep, init_sparsity=0.15, seed=0,
    )
    H_A_sw, _ = run_ca_stateful(
        np.concatenate([Z_shared, Z_A_tail], axis=0),
        conn_sw, rule_sweep, init_sparsity=0.15, seed=0,
    )
    hamming = (H_A_sw != H_B_sw).mean(axis=1)
    ax.plot(hamming, lw=1.5, color="#d7191c")
    ax.axvline(T_SPLIT, color="black", ls="--", lw=1.0)
    m_ = compute_metrics(H_sw, Z_iid)
    ax.set_title(
        f"α_h={alpha_h}\n"
        f"sparsity={m_['mean_sparsity']:.3f}  "
        f"Δh={m_['mean_hamming_dist']:.3f}",
        fontsize=9,
    )
    ax.set_xlabel("Time step", fontsize=9)

axes[0].set_ylabel("Hamming distance", fontsize=9)
fig.suptitle(
    "MI Evidence: memory timescale vs separation quality (α_h sweep)",
    fontsize=11,
)
plt.tight_layout()
plt.show()